<a href="https://colab.research.google.com/github/niranjansendilkumar11/Irish-weather-pipeline/blob/main/ca2_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import os
import json
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import requests



In [11]:
#API Configuration
from google.colab import userdata

try:
  API_KEY =userdata.get('OWM_API_KEY')
  print("API Key loaded successfully!")
except Exception:
  API_KEY = ""


API Key loaded successfully!


In [12]:
IRISH_CITIES = [
    "Dublin,IE",
    "Cork,IE",
    "Galway,IE",
    "Limerick,IE",
    "Waterford,IE",
    "Swords,IE",
    "Sligo,IE",
    "Derry,IE",
    "Drogheda,IE",
    "Tallaght,IE"
]
print(f"Monitoring {len(IRISH_CITIES)} Irish cities:")
for city in IRISH_CITIES:
    print(f"{city.split(',')[0]}")


Monitoring 10 Irish cities:
Dublin
Cork
Galway
Limerick
Waterford
Swords
Sligo
Derry
Drogheda
Tallaght


In [13]:
import time

BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

def fetch_city_weather(city_name):
    params = {
        "q": city_name,
        "appid": API_KEY,
        "units": "metric"
    }
    try:
      response = requests.get(BASE_URL, params=params,timeout=10)
      if response.status_code == 200:
        data = response.json()
        return {
                "city":          data["name"],
                "country":       data["sys"]["country"],
                "latitude":      data["coord"]["lat"],
                "longitude":     data["coord"]["lon"],
                "temp_celsius":  data["main"]["temp"],
                "feels_like":    data["main"]["feels_like"],
                "temp_min":      data["main"]["temp_min"],
                "temp_max":      data["main"]["temp_max"],
                "humidity":      data["main"]["humidity"],
                "pressure":      data["main"]["pressure"],
                "visibility":    data.get("visibility", None),
                "wind_speed":    data["wind"]["speed"],
                "wind_degree":   data["wind"].get("deg", None),
                "weather_main":  data["weather"][0]["main"],
                "weather_desc":  data["weather"][0]["description"],
                "cloud_coverage":data["clouds"]["all"],
                "sunrise":       data["sys"]["sunrise"],
                "sunset":        data["sys"]["sunset"],
                "fetched_at":    datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
            }
      else:
            print(f"Failed for {city_name} - Status: {response.status_code}")
            return None
    except requests.exceptions.Timeout:
        print(f"Timeout for {city_name}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Error for {city_name}: {e}")
        return None


In [14]:
result = fetch_city_weather ("Dublin,IE")
print(result)

{'city': 'Dublin', 'country': 'IE', 'latitude': 53.344, 'longitude': -6.2672, 'temp_celsius': 5.08, 'feels_like': 1.65, 'temp_min': 5.08, 'temp_max': 5.08, 'humidity': 77, 'pressure': 1020, 'visibility': 10000, 'wind_speed': 4.57, 'wind_degree': 283, 'weather_main': 'Clouds', 'weather_desc': 'scattered clouds', 'cloud_coverage': 25, 'sunrise': 1774419349, 'sunset': 1774464372, 'fetched_at': '2026-03-25 20:01:11'}


In [15]:
#Section 4 - Data Acquisition
# This section fetches live weather data for all 10 Irish cities
def fetch_all_cities(cities):
  all_records = []
  for city in cities:
    print(f"Fetching {city.split(',')[0]}...")
    record = fetch_city_weather(city)
    if record:
      all_records.append(record)
    time.sleep(1)
  df_raw = pd.DataFrame(all_records)
  print(f"Done Fetched {len(all_records)} cities")
  return df_raw

df_raw = fetch_all_cities(IRISH_CITIES)


Fetching Dublin...
Fetching Cork...
Fetching Galway...
Fetching Limerick...
Fetching Waterford...
Fetching Swords...
Fetching Sligo...
Fetching Derry...
Fetching Drogheda...
Fetching Tallaght...
Done Fetched 10 cities


In [16]:
# Displaying the raw  data we fetched
df_raw.head(10)

,city,country,latitude,longitude,temp_celsius,feels_like,temp_min,temp_max,humidity,pressure,visibility,wind_speed,wind_degree,weather_main,weather_desc,cloud_coverage,sunrise,sunset,fetched_at
0,Dublin,IE,53.3440,-6.2672,5.08,1.65,5.08,5.08,77,1020,10000,4.57,283,Clouds,scattered clouds,25,1774419349,1774464372,2026-03-25 20:01:11
1,Cork,IE,51.8980,-8.4706,5.30,2.09,5.30,5.30,72,1024,10000,4.25,310,Rain,light rain,44,1774419917,1774464861,2026-03-25 20:01:12
2,Galway,IE,53.2719,-9.0489,5.58,1.88,5.58,5.58,64,1024,10000,5.41,297,Clouds,overcast clouds,97,1774420018,1774465039,2026-03-25 20:01:13
3,Limerick,IE,52.6647,-8.6231,5.43,1.36,5.43,5.43,69,1024,10000,6.20,309,Clouds,broken clouds,65,1774419933,1774464919,2026-03-25 20:01:14
4,Waterford,IE,52.2583,-7.1119,5.78,2.10,5.78,5.78,66,1023,10000,5.48,309,Clear,clear sky,7,1774419582,1774464545,2026-03-25 20:01:15
5,Swords,IE,53.4597,-6.2181,5.23,1.36,5.23,5.23,76,1020,10000,5.58,289,Clouds,few clouds,23,1774419334,1774464364,2026-03-25 20:01:16
6,Sligo,IE,54.2697,-8.4694,5.57,0.38,5.57,5.57,75,1021,10000,9.87,309,Clouds,overcast clouds,93,1774419849,1774464929,2026-03-25 20:01:17
7,Derry,IE,51.5867,-9.0503,5.24,1.14,5.24,5.24,71,1026,10000,6.16,309,Clouds,scattered clouds,31,1774420065,1774464992,2026-03-25 20:01:18
8,Drogheda,IE,53.7189,-6.3478,5.08,0.70,5.08,5.08,75,1020,10000,6.76,294,Clouds,few clouds,21,1774419357,1774464403,2026-03-25 20:01:19
9,Tallaght,IE,53.2859,-6.3734,4.69,1.24,4.69,4.69,78,1021,10000,4.43,286,Rain,moderate rain,26,1774419376,1774464396,2026-03-25 20:01:20


In [17]:
#section 5- Transformations
# This section cleans and enriches the raw weather data

def categorise_temperature (temp_celsius):
  if temp_celsius < 0:
    return "Freezing"
  elif temp_celsius < 8:
        return "Cold"
  elif temp_celsius < 14:
        return "Cool"
  elif temp_celsius < 19:
        return "Mild"
  else:
        return "Warm"

In [18]:
def categorise_wind(wind_speed):
    if wind_speed < 3.0:
        return "Calm"
    elif wind_speed < 8.0:
        return "Breezy"
    elif wind_speed < 14.0:
        return "Windy"
    else:
        return "Storm"


In [19]:
def categorise_humidity(humidity):
  if humidity < 40:
        return "Dry"
  elif humidity < 70:
        return "Comfortable"
  else:
        return "Humid"

In [20]:
#calculating how many hours of daylight each city gets
def convert_unix_timestamp(unix_time):
    return datetime.fromtimestamp(unix_time).strftime('%H:%M:%S')

def calculate_daylight_hours(sunrise_unix, sunset_unix):
    duration_seconds = sunset_unix - sunrise_unix
    return round(duration_seconds / 3600, 2)

print(convert_unix_timestamp(1774333095))
print(calculate_daylight_hours(1774333095, 1774377864))

06:18:15
12.44


In [21]:
# Applies all transformations to the raw dataframe and returns a cleaned enriched version
def transform_weather_data(df):
  df_transformed = df.copy()

  df_transformed["sunrise_time"] = df_transformed["sunrise"].apply(convert_unix_timestamp)
  df_transformed["sunset_time"] = df_transformed["sunset"].apply(convert_unix_timestamp)
  df_transformed["daylight_hours"] = df_transformed.apply(
        lambda row: calculate_daylight_hours(row["sunrise"], row["sunset"]), axis=1
    )
  df_transformed["temp_category"] = df_transformed["temp_celsius"].apply(categorise_temperature)
  df_transformed["wind_category"] = df_transformed["wind_speed"].apply(categorise_wind)
  df_transformed["humidity_category"] = df_transformed["humidity"].apply(categorise_humidity)
  df_transformed["visibility_km"] = (df_transformed["visibility"] / 1000).round(2)
  df_transformed.drop(columns=["sunrise", "sunset", "visibility"], inplace=True)

  return df_transformed

df_transformed = transform_weather_data(df_raw)
print("Transformations applied successfully!")
print(f"Columns now: {df_transformed.shape[1]}")
df_transformed.head(10)

Transformations applied successfully!
Columns now: 23


,city,country,latitude,longitude,temp_celsius,feels_like,temp_min,temp_max,humidity,pressure,...,weather_desc,cloud_coverage,fetched_at,sunrise_time,sunset_time,daylight_hours,temp_category,wind_category,humidity_category,visibility_km
0,Dublin,IE,53.3440,-6.2672,5.08,1.65,5.08,5.08,77,1020,...,scattered clouds,25,2026-03-25 20:01:11,06:15:49,18:46:12,12.51,Cold,Breezy,Humid,10.0
1,Cork,IE,51.8980,-8.4706,5.30,2.09,5.30,5.30,72,1024,...,light rain,44,2026-03-25 20:01:12,06:25:17,18:54:21,12.48,Cold,Breezy,Humid,10.0
2,Galway,IE,53.2719,-9.0489,5.58,1.88,5.58,5.58,64,1024,...,overcast clouds,97,2026-03-25 20:01:13,06:26:58,18:57:19,12.51,Cold,Breezy,Comfortable,10.0
3,Limerick,IE,52.6647,-8.6231,5.43,1.36,5.43,5.43,69,1024,...,broken clouds,65,2026-03-25 20:01:14,06:25:33,18:55:19,12.50,Cold,Breezy,Comfortable,10.0
4,Waterford,IE,52.2583,-7.1119,5.78,2.10,5.78,5.78,66,1023,...,clear sky,7,2026-03-25 20:01:15,06:19:42,18:49:05,12.49,Cold,Breezy,Comfortable,10.0
5,Swords,IE,53.4597,-6.2181,5.23,1.36,5.23,5.23,76,1020,...,few clouds,23,2026-03-25 20:01:16,06:15:34,18:46:04,12.51,Cold,Breezy,Humid,10.0
6,Sligo,IE,54.2697,-8.4694,5.57,0.38,5.57,5.57,75,1021,...,overcast clouds,93,2026-03-25 20:01:17,06:24:09,18:55:29,12.52,Cold,Windy,Humid,10.0
7,Derry,IE,51.5867,-9.0503,5.24,1.14,5.24,5.24,71,1026,...,scattered clouds,31,2026-03-25 20:01:18,06:27:45,18:56:32,12.48,Cold,Breezy,Humid,10.0
8,Drogheda,IE,53.7189,-6.3478,5.08,0.70,5.08,5.08,75,1020,...,few clouds,21,2026-03-25 20:01:19,06:15:57,18:46:43,12.51,Cold,Breezy,Humid,10.0
9,Tallaght,IE,53.2859,-6.3734,4.69,1.24,4.69,4.69,78,1021,...,moderate rain,26,2026-03-25 20:01:20,06:16:16,18:46:36,12.51,Cold,Breezy,Humid,10.0


In [23]:
# Section 6 - Unit Testing
#Testing each transformation function to verify they return correct results
import unittest

class TestWeatherData(unittest.TestCase):

  def test_categorise_temperature(self):
    self.assertEqual(categorise_temperature(-5), "Freezing")
    self.assertEqual(categorise_temperature(4), "Cold")
    self.assertEqual(categorise_temperature(10), "Cool")
    self.assertEqual(categorise_temperature(16), "Mild")
    self.assertEqual(categorise_temperature(22), "Warm")

  def test_categorise_wind(self):
    self.assertEqual(categorise_wind(1.5), "Calm")
    self.assertEqual(categorise_wind(5.0), "Breezy")
    self.assertEqual(categorise_wind(10.0), "Windy")
    self.assertEqual(categorise_wind(18.0), "Storm")

  def test_categorise_humidity(self):
    self.assertEqual(categorise_humidity(30), "Dry")
    self.assertEqual(categorise_humidity(55), "Comfortable")
    self.assertEqual(categorise_humidity(85), "Humid")

  def test_calculate_daylight_hours(self):
    self.assertEqual(calculate_daylight_hours(1774333095, 1774377864), 12.44)

unittest.main(argv=[""], exit=False, verbosity=2)

test_calculate_daylight_hours (__main__.TestWeatherData.test_calculate_daylight_hours) ... ok
test_categorise_humidity (__main__.TestWeatherData.test_categorise_humidity) ... ok
test_categorise_temperature (__main__.TestWeatherData.test_categorise_temperature) ... ok
test_categorise_wind (__main__.TestWeatherData.test_categorise_wind) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.010s

OK
